# 04. Bigram Language Model — 最小の言語モデル

## 学習目標

- 「言語モデル＝次トークンの条件付き分布」を最小構成で確認する
- **カウント（閉形式MLE）とSGD学習が同じ解に到達する**ことを実測で見る
- 平滑化ハイパーパラメータ α が損失を大きく動かすことを知る

## 数式

bigram モデルは直前1トークンだけで次を予測する:

$$P(x_{t+1}=j \mid x_t=i) = p_{ij}, \qquad
\hat p_{ij} = \frac{c_{ij} + \alpha}{\sum_k c_{ik} + \alpha V}$$

$c_{ij}$ は訓練データ中の遷移 $i \to j$ の回数、$\alpha$ は加算平滑化（未観測ペアに残す確率質量）。
損失は次トークンの負対数尤度（nats/token）。**一様分布なら $\ln V$**。

## 手計算例

コーパス「あいあいう」（遷移: あ→い, い→あ, あ→い, い→う）、語彙 {あ,い,う}, α=0 の場合:

| from\to | あ | い | う | 行和 |
|---|---|---|---|---|
| あ | 0 | 2 | 0 | 2 |
| い | 1 | 0 | 1 | 2 |
| う | 0 | 0 | 0 | 0 |

→ $P(\text{い}\mid\text{あ}) = 2/2 = 1.0$、$P(\text{あ}\mid\text{い}) = P(\text{う}\mid\text{い}) = 0.5$。
「う」の行は観測ゼロ → α>0 が無いと定義できない（平滑化の必然性）。

In [1]:
# 共通セットアップ（全ノートブック同一）
import warnings
warnings.filterwarnings("ignore")

import math
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"ROOT={ROOT}")
print(f"device={DEVICE}, torch={torch.__version__}")

ROOT=/home/kazumasa/projects/jp_llm_lab
device=cuda, torch=2.11.0+cu128


In [2]:
# 実行済み run の成果物を読む（再学習不要）
BRUN = ROOT / "experiments/runs/m1_bigram_char_seed42"
assert BRUN.exists(), "先に `make -C jp_llm_lab train-bigram` を実行してください"
recs = read_jsonl(BRUN / "metrics.jsonl")
summary = load_json(BRUN / "summary.json")

fig = go.Figure()
fig.add_trace(go.Scatter(x=[r["step"] for r in recs], y=[r["val_loss"] for r in recs],
                         name="neural bigram (val)", line=dict(color="#ff7f0e")))
fig.add_hline(y=summary["count"]["val_loss"], line_dash="dash", line_color="#7f7f7f",
              annotation_text=f"count model (val {summary['count']['val_loss']:.3f})")
fig.add_hline(y=summary["uniform_loss_reference"], line_dash="dot", line_color="#bbb",
              annotation_text=f"uniform ln V = {summary['uniform_loss_reference']:.2f}")
fig.update_layout(title="neural bigram は count モデル（閉形式解）に収束する",
                  xaxis_title="SGD step", yaxis_title="val loss (nats/token)",
                  template="plotly_white", height=420)
fig.show()
print(f"neural val = {summary['neural']['val_loss']:.4f} / count val = {summary['count']['val_loss']:.4f}")

neural val = 3.5407 / count val = 3.5919


In [3]:
# 遷移確率行列を「見る」: 頻出30文字 × 頻出30文字
from jp_llm_lab.data.sample_corpus import load_sample_corpus
from jp_llm_lab.models.bigram import CountBigramLM
from jp_llm_lab.tokenization.char_tokenizer import CharTokenizer

corpus = load_sample_corpus("kokoro")
tok = CharTokenizer.train([corpus])
ids = torch.tensor(tok.encode(corpus), dtype=torch.long)
cm = CountBigramLM(tok.vocab_size, alpha=0.05).fit(ids)

top_ids = [i for i, _ in Counter(ids.tolist()).most_common(30)]
labels = [tok.id_to_token(i).replace("\n", "⏎").replace("　", "␣") for i in top_ids]
sub = cm.log_probs.exp()[top_ids][:, top_ids]
fig = go.Figure(go.Heatmap(z=sub.tolist(), x=labels, y=labels, colorscale="Blues",
                           hovertemplate="P(%{x} | %{y}) = %{z:.3f}<extra></extra>"))
fig.update_layout(title="bigram 遷移確率（行=現在の文字, 列=次の文字）",
                  template="plotly_white", height=560)
fig.update_yaxes(autorange="reversed")
fig.show()

**How to read**: 行「し」で「た」が明るい＝「し→た」が高確率、のように読む。
「。」の行は「⏎（改行）」や「」」が明るいはず — 句点の後の統計が既に captureされている。

In [4]:
# 平滑化 α の感度（summary に保存済みのスイープ）
sweep = summary["alpha_sweep_val_loss"]
fig = go.Figure(go.Scatter(x=[float(a) for a in sweep], y=list(sweep.values()),
                           mode="lines+markers", line=dict(color="#7f7f7f")))
fig.add_hline(y=summary["neural"]["val_loss"], line_dash="dash", line_color="#ff7f0e",
              annotation_text="neural bigram")
fig.update_xaxes(type="log", title="α (log)")
fig.update_layout(title="加算平滑化 α と検証損失", yaxis_title="val loss (nats/token)",
                  template="plotly_white", height=400)
fig.show()

In [5]:
# bigram からのサンプリング — 「1文字先の統計」だけでどこまで日本語に見えるか
g = torch.Generator().manual_seed(0)
sample_ids = cm.generate(tok.encode("私")[0], 120, g)
print(tok.decode(sample_ids))

私の室を忘れです。
　更茂修峙虚統欠移危詐りゃあり離騒箒際べ確点胆嬌鈍傷顋を掛けしていわ東俥摺焚掻か知形鄲机の厭だけまして驚ケツ芽監這ぱ推住都業模閻扉細樹蒲裕方がで賛弧コ吸婚出犬閃創響衰宗坪働看力糖拠茸学科憚を増連熊重配段奪悵泊った。実達手紙


## Observation / Interpretation / Caveat

- **Observation**: neural bigram の val loss は count モデルとほぼ一致（差 ~0.05 nats は平滑化方式の差）。
  α は 1桁違うだけで val loss を大きく動かし、α=0.5 は明確に悪い（α·V ≈ 1000 擬似カウントが実カウント≈73/行を圧倒）。
  サンプルは局所的には日本語らしいが、文としては即座に破綻する。
- **Interpretation**: 「ニューラル＝賢い」のではなく、**表現力が同じなら学習法が違っても同じ解**。
  Transformer の価値は最適化ではなく「1トークンより長い文脈を条件にできる」表現力にある。
- **Caveat**: この一致は bigram という凸な問題だから成立する。深いモデルでは
  最適化の経路・初期化が解に影響する（M3 §14.3）。

## 確認問題

1. 一様分布の損失が $\ln V$ になることを導け。
2. α→∞ のとき count モデルはどんな分布に近づくか。
3. bigram が原理的に表現できない日本語の現象を1つ挙げよ（例: 係り受け）。

**次へ**: [05_attention_from_scratch](05_attention_from_scratch.ipynb) — 文脈を「選んで」参照する仕組み。